In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from gensim.models import Word2Vec
import re
import warnings
import random
import os
import tempfile
warnings.filterwarnings("ignore") # ignoring all warnings that the code is faced

# Preprocessing the text by converting to lowercase, removing non-alphanumeric characters & handling numbers as a placeholder word
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)# Removing non-alphanumeric characters
    text = re.sub(r"\s+", " ", text).strip() # Replacing multiple or all white spaces with a single space
    words = re.findall(r'\b\w+\b', text)
    # Replacing any number (integer or with decimal) with the placeholder "numbers"
    for index, w in enumerate(words):
        if w.replace('.', '', 1).isdigit() or re.search(r'\d', w):
            words[index] = "numbers"
    return words

def load_data(file_path):
    try:
        return pd.read_csv(file_path)
    except pd.errors.ParserError:
        # Fallbacking to alternative separator
        return pd.read_csv(file_path, sep='|', on_bad_lines='skip')

def extract_text_column(df):
    col = df.columns[0] # Like past codes, assuming the first column contains text data
    sample = df[col].iloc[0]
    if isinstance(sample, str) and ',' in sample:
        df[col] = df[col].apply(lambda x: ' '.join(x.split(',')) if isinstance(x, str) else x)
    return df[col].astype(str).tolist()

def custom_ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

# Part 1
def generate_ngrams(corpus, n):
    sequences = []
    for sentence in corpus:
        tokens = preprocess(sentence)
    # Skipping sentences that are shorter than n due to the limitation of model
        if len(tokens) < n: 
            continue
        for gram in custom_ngrams(tokens, n):
            sequences.append(gram)
    return sequences

def train_word2vec(corpus):
    tokenized = [preprocess(sent) for sent in corpus if preprocess(sent)]  
    if not tokenized:
        raise ValueError("Empty corpus after preprocessing")
    model = Word2Vec(sentences=tokenized, vector_size=100, window=5, min_count=1, workers=4)
    return model

# Creating input and target tensors for training
def create_dataset(ngrams_list, w2v_model):
    inputs, targets = [], []
    for gram in ngrams_list:
        # Checking that all tokens are in Word2Vec model
        if all(token in w2v_model.wv for token in gram):
            input_vec = np.concatenate([w2v_model.wv[token] for token in gram[:-1]])
            target_vec = w2v_model.wv[gram[-1]] # Using the last token's vector as target
            inputs.append(input_vec)
            targets.append(target_vec)
    if not inputs:
        return None, None  
    return torch.tensor(inputs, dtype=torch.float32), torch.tensor(targets, dtype=torch.float32)

# Part 2
class FeedForwardNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(FeedForwardNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)# Fully connected layer
        self.relu = nn.ReLU() # ReLU activation
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)

# Train the neural network model using MSE loss and Adam optimizer
def train_model(model, train_X, train_y, lr=0.01, epochs=10):
    if train_X is None or train_y is None:
        return None  
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        output = model(train_X)
        loss = criterion(output, train_y)
        if torch.isnan(loss) or torch.isinf(loss):
            return None  
        loss.backward()
        optimizer.step()
    return model

def evaluate_perplexity(model, test_X, test_y):
    # Checking for returning infinity if data or model is missing
    if model is None or test_X is None or test_y is None:
        return float('inf')  
    model.eval()
    with torch.no_grad():
        predictions = model(test_X)
        loss = nn.MSELoss()(predictions, test_y)
        if torch.isnan(loss) or torch.isinf(loss):
            return float('inf')
        return torch.exp(loss.clamp(min=-10, max=10)).item() 

# Part 3
def run_pipeline(train_text, valid_text, test_text):
    w2v = train_word2vec(train_text)
    # Cosicering our best model with its parameters
    n = 3
    lr = 0.01
    hidden_dim = 128
    epochs = 30

    train_ngrams = generate_ngrams(train_text, n)
    valid_ngrams = generate_ngrams(valid_text, n)
    test_ngrams = generate_ngrams(test_text, n)

    train_X, train_y = create_dataset(train_ngrams, w2v)
    valid_X, valid_y = create_dataset(valid_ngrams, w2v)
    test_X, test_y = create_dataset(test_ngrams, w2v)

    if train_X is None or valid_X is None or test_X is None:
        print("Failed: Empty dataset for trigram model")
        return w2v, n, lr, None

    input_dim = train_X.shape[1]
    output_dim = train_y.shape[1]
    model = FeedForwardNN(input_dim, hidden_dim, output_dim)
    model = train_model(model, train_X, train_y, lr=lr, epochs=epochs)
    if model is None:
        print("Failed: Trigram model training resulted in invalid loss")
        return w2v, n, lr, None
    valid_perp = evaluate_perplexity(model, valid_X, valid_y)
    test_perp = evaluate_perplexity(model, test_X, test_y)

    print(f"\nTrigram FFNN Validation Perplexity: {valid_perp:.2f}")
    print(f"Trigram FFNN Test Perplexity: {test_perp:.2f}")
    print(f"Model Parameters: Trigram (LR={lr}, HiddenDim={hidden_dim}, Epochs={epochs})")

    return w2v, n, lr, model

# Part 4 
# Predicting the next words in given sentences
def predict_next_words(model, w2v_model, sentences, n):
    if model is None:
        return [(sentence, "", "") for sentence in sentences]  
    predictions = []
    for sentence in sentences:
        tokens = preprocess(sentence)
        temp_tokens = tokens[:]
        generated = []
        max_words = random.randint(1, 4)
        for _ in range(max_words):
            if len(temp_tokens) < n - 1:
                break
            context = temp_tokens[-(n - 1):] # Getting the last n-1 tokens as the context
            # Making sure that all context words are in Word2Vec model
            if not all(word in w2v_model.wv for word in context):
                break
            input_vec = torch.tensor(np.concatenate([w2v_model.wv[word] for word in context])).float()
            pred_vec = model(input_vec)
            next_word = w2v_model.wv.similar_by_vector(pred_vec.detach().numpy(), topn=1)[0][0] # Getting to the next predicted word
            if next_word == "numbers":
                next_word = str(random.randint(1, 15)) # Replacing "numbers" with a random integer
            # Avoiding repeating the last predicted word
            if len(generated) > 0 and next_word == generated[-1]:
                break
            generated.append(next_word)
            temp_tokens.append(next_word)
        first_pred = generated[0] if generated else ""
        if first_pred == "numbers":
            first_pred = str(random.randint(1, 15)) # Replacing "numbers" with a random integer
        full_pred = " ".join(str(random.randint(1, 15)) if word == "numbers" else word for word in generated) if generated else ""
        predictions.append((sentence, first_pred, full_pred))

    return predictions

# Part 5
def main():
    base_path = "E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/"
    train_df = load_data(base_path + "train.csv")
    valid_df = load_data(base_path + "val.csv")
    test_df = load_data(base_path + "test.csv")
    train_text = extract_text_column(train_df)
    valid_text = extract_text_column(valid_df)
    test_text = extract_text_column(test_df)
    w2v, best_n, best_lr, model = run_pipeline(train_text, valid_text, test_text)
    sample_df = load_data(base_path + "sample.csv")
    sample_text = extract_text_column(sample_df)
    predictions = predict_next_words(model, w2v, sample_text, best_n)
    cleaned_predictions = []
    for pred in predictions:
        incomplete_text = str(pred[0]).replace(',', ' ').replace('\n', ' ')
        first_pred = str(pred[1]).replace(',', ' ').replace('\n', ' ')
        full_pred = str(pred[2]).replace(',', ' ').replace('\n', ' ')
        cleaned_predictions.append((incomplete_text, first_pred, full_pred))
    output_df = pd.DataFrame(cleaned_predictions, columns=['incomplete_text', 'first_pred', 'full_pred'])
    output_path = base_path + "sample_predictions.xlsx"
    try:
        output_df.to_excel(output_path, index=False, engine='openpyxl')
        print(f"Successfully saved predictions to {output_path}")
    except PermissionError:
        print(f"PermissionError: Could not write to {output_path}. The file may be open in another program (e.g., Excel). Please close it and try again.")
        try:
            with tempfile.NamedTemporaryFile(mode='wb', delete=False, suffix='.xlsx', dir=base_path) as temp_file:
                output_df.to_excel(
                    temp_file.name,
                    index=False,
                    engine='openpyxl'
                )
            os.replace(temp_file.name, output_path)
            print(f"Successfully saved predictions to {output_path} using temporary file")
        except Exception as e:
            print(f"Failed to save predictions: {e}")
if __name__ == "__main__":
    main()


Trigram FFNN Validation Perplexity: 2.13
Trigram FFNN Test Perplexity: 2.16
Model Parameters: Trigram (LR=0.01, HiddenDim=128, Epochs=30)
Successfully saved predictions to E:/Aidin/Universities/Amirkabir (Master)/Lessons/My-Terms/Term 3/Natural Language Processing/Assignments/HW1/Question/Dataset/sample_predictions.xlsx
